# Devixo Solutions — AI/ML Internship Program
## Task 01: Python Programming, Data Analysis & Visualization

**Project:** Phishing Website Detection — Exploratory Data Analysis of 30 Security Features

| | |
|---|---|
| **Student Name** | Shahlal Jaffir |
| **Internship Domain** | AI/ML — Cybersecurity |
| **Environment** | Jupyter Notebook (Python 3) |
| **Submission Date** | 22 November 2026 |
| **Dataset** | Phishing Websites (Mohammad & McCluskey, 2012) — UCI ML Repository / Kaggle |
| **License** | CC BY 4.0 |

---

## 1. Project Introduction

Phishing remains one of the most common entry points for cyber attacks. Instead of
attacking a network directly, a phishing site impersonates a trusted brand and
persuades a user to hand over credentials. Because the attack surface is the
**web page itself**, many phishing sites can be recognised from measurable
properties of their URL, domain and page structure — for example whether the URL
uses an IP address instead of a domain name, whether the SSL certificate is
valid, or whether anchor links point outside the site.

This notebook analyses a real, publicly available dataset of **11,055 websites**,
each described by **30 such security features**, and performs a complete data
analysis pipeline: loading, understanding, cleaning, exploratory data analysis,
descriptive statistics, correlation analysis, ten visualizations, and
evidence-based conclusions.

The goal is not to build a classifier (that belongs to a later task). The goal is
to understand the data, clean it honestly, quantify which features separate
phishing sites from legitimate ones, and communicate the findings clearly.

## 2. Problem Statement

Security teams receive far more suspicious URLs than they can inspect manually.
Before any automated detection model can be built, two questions must be answered
from data:

1. **What does the data actually contain, and how trustworthy is it?**
   (structure, duplicates, missing values, encoding, invalid entries)
2. **Which measurable website properties differ meaningfully between phishing
   and legitimate sites?**

Without this analysis, any later machine-learning model would be built on
unverified assumptions. This task therefore addresses the problem:

> *Given a dataset of 30 URL/domain/page-level security features for 11,055
> websites, clean the data, characterise it statistically, and determine which
> features carry genuine discriminative signal between phishing and legitimate
> sites.*

## 3. Objectives

1. Load the dataset with Pandas and inspect its structure (rows, columns, types).
2. Classify every column by type and explain why the classification matters.
3. Detect and handle missing values, duplicates, incorrect data types, unclear
   column names and invalid values — with justification for every decision.
4. Produce descriptive statistics (mean, median, mode, min, max, range, standard
   deviation, variance) and interpret them.
5. Quantify relationships between features using a correlation matrix.
6. Produce the ten required visualizations, each with title, axis labels and an
   interpretation of what it shows.
7. Extract at least ten insights that are supported by the actual numbers.
8. Summarise the analysis in a conclusion and document everything reproducibly.

## 4. Dataset Description and Source

| Property | Value |
|---|---|
| **Name** | Phishing Websites |
| **Authors** | Rami M. Mohammad, Lee McCluskey (University of Huddersfield) |
| **Year** | 2012 (donated to UCI 2015) |
| **Original repository** | UCI Machine Learning Repository — dataset ID 327 |
| **DOI** | 10.24432/C51W2X |
| **Kaggle mirrors** | `akashkr/phishing-website-dataset`, `isatish/phishing-dataset-uci-ml-csv`, `odixe0502/uci-phishing-websites-onehot` |
| **License** | Creative Commons Attribution 4.0 International (CC BY 4.0) |
| **Instances** | 11,055 |
| **Features** | 30 predictive + 1 target (`Result`) |
| **Feature type** | Integer-coded categorical: `-1`, `0`, `1` |
| **Missing values (original)** | None reported |
| **Task type** | Binary classification (phishing vs legitimate) |

**How the data was collected (per the authors):** the dataset was assembled from
the PhishTank archive, the MillerSmiles archive and Google search operators. Each
feature encodes a rule-based assessment of one property of a website:

* `-1` — property indicates a **phishing** pattern
* `0` — **suspicious / inconclusive**
* `1` — property indicates a **legitimate** pattern

Target column `Result`: `-1` = phishing, `1` = legitimate.

> **Provenance note.** This notebook loads the CSV committed in the repository
> (`data/phishing_dataset.csv`), which is a direct conversion of the official UCI
> `Training Dataset.arff`. If the CSV is absent, the notebook downloads the
> original archive from UCI and parses it automatically, so the analysis stays
> reproducible. No value has been altered.

## 5. Importing Libraries

* **Pandas** — loading, cleaning, grouping and tabular analysis
* **NumPy** — numerical operations and vectorised calculations
* **Matplotlib** — base plotting engine
* **Seaborn** — statistical visualizations built on Matplotlib

In [ ]:
import io
import os
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# ---- global plotting style (applied to every chart below) ----
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.autolayout": True,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

RANDOM_STATE = 42

print("pandas     :", pd.__version__)
print("numpy      :", np.__version__)
print("matplotlib :", plt.matplotlib.__version__)
print("seaborn    :", sns.__version__)

In [ ]:
FIG_DIR = "visualizations"
os.makedirs(FIG_DIR, exist_ok=True)
_saved_figures = []


def save_figure(fig, name):
    """Save a figure as PNG and remember it for the export step."""
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    _saved_figures.append(path)
    print(f"saved -> {path}")


print("Figure export directory:", os.path.abspath(FIG_DIR))


## 6. Dataset Loading

The loader looks for the CSV in several locations (repository layout and Colab
upload), and falls back to downloading the official UCI archive and parsing the
ARFF file if the CSV is not found.

In [ ]:
CANDIDATE_PATHS = [
    "data/phishing_dataset.csv",              # repository layout
    "../data/phishing_dataset.csv",
    "data/phishing_dataset.csv",          # Colab upload
    "phishing_dataset.csv",
]

DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)


def load_from_uci():
    """Download the official UCI archive and parse the ARFF file."""
    url = "https://archive.ics.uci.edu/static/public/327/phishing+websites.zip"
    raw = urllib.request.urlopen(url, timeout=120).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    member = next(n for n in zf.namelist()
                  if n.lower().endswith(".arff") and "training" in n.lower())
    lines = zf.read(member).decode("utf-8", errors="ignore").splitlines()

    attrs, start = [], None
    for i, line in enumerate(lines):
        s = line.strip()
        if s.lower().startswith("@attribute"):
            attrs.append(s.split()[1].strip("'\""))
        elif s.lower().startswith("@data"):
            start = i + 1
            break

    records = [l.strip().split(",") for l in lines[start:] if l.strip()]
    return pd.DataFrame(records, columns=attrs).astype(int)


if DATA_PATH:
    df_raw = pd.read_csv(DATA_PATH)
    print(f"Loaded from: {DATA_PATH}")
else:
    df_raw = load_from_uci()
    print("Downloaded and parsed from UCI archive")

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

## 7. Initial Inspection

### 7.1 First 5, last 5 and a random sample

In [ ]:
print("FIRST 5 RECORDS")
display(df_raw.head())

print("LAST 5 RECORDS")
display(df_raw.tail())

print("RANDOM SAMPLE (5 records, random_state=42)")
display(df_raw.sample(5, random_state=RANDOM_STATE))

### 7.2 Structure of the dataset

In [ ]:
print("DATASET SHAPE")
print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]}")
print()

print("COLUMN NAMES")
for i, c in enumerate(df_raw.columns, 1):
    print(f"{i:2d}. {c}")
print()

print("DATA TYPES / NON-NULL COUNTS")
df_raw.info()

### 7.3 Shape and basic numeric summary

In [ ]:
print("Shape:", df_raw.shape)

display(df_raw.describe().T.round(3))

# how many distinct values each column takes
print("\nDistinct values per column (all should be 3 or fewer):")
print(df_raw.nunique().sort_values().to_string())

## 8. Column Classification

Every column in this dataset is stored as an integer, but **numerically encoded
categorical data is still categorical**. The distinction matters:

* **Numerical** variables support arithmetic (mean, standard deviation) and
  scatter/line plots.
* **Categorical** variables support counts, proportions and bar/pie/count plots.
* **Date/time** variables support trend analysis.

Treating an encoded category as a number without care produces meaningless
averages — for example, the "average" of `-1/0/1` codes has no physical meaning
by itself. In this analysis the codes are treated as **ordinal** values
(`-1 < 0 < 1`), which is why means such as "mean SSL state = -0.48 for phishing"
are meaningful as *direction and strength of a signal* rather than as real units.

In [ ]:
# Identify columns whose values are limited to a small set of codes
code_columns = [c for c in df_raw.columns if df_raw[c].nunique() <= 3]

numerical_cols = []      # no continuous measurements exist in this dataset
categorical_cols = code_columns
datetime_cols = [c for c in df_raw.columns if pd.api.types.is_datetime64_any_dtype(df_raw[c])]

print(f"Numerical (continuous) columns : {len(numerical_cols)}")
print(f"Categorical / ordinal columns  : {len(categorical_cols)}")
print(f"Date/time columns              : {len(datetime_cols)}")
print()
print("Target column : 'Result'  ->  -1 = Phishing, 1 = Legitimate")
print()
print("Categorical (ordinal-coded) columns:")
for c in categorical_cols:
    print(f"  - {c:30s} values: {sorted(df_raw[c].unique())}")

**Interpretation.** There are no continuous measurements (no prices, ages or
durations) — every feature is a three-level ordinal code. Consequently:

* Bar charts, count plots, pie charts and heatmaps are the natural tools.
* A **derived numeric feature** — the number of phishing-leaning signals per
  website — is created later so that histograms, box plots, violin plots,
  scatter plots and line charts describe something real rather than a single
  ternary code.

## 9. Missing-Value Analysis

In [ ]:
missing = pd.DataFrame({
    "Missing_Count": df_raw.isna().sum(),
    "Missing_Percent": (df_raw.isna().mean() * 100).round(2),
})
missing = missing.sort_values("Missing_Count", ascending=False)

print(f"Total cells      : {df_raw.size:,}")
print(f"Total missing    : {int(df_raw.isna().sum().sum()):,}")
print(f"Columns affected : {(missing['Missing_Count'] > 0).sum()}")
print()
display(missing.T)

### 9.1 Handling decision

**Result: no missing values exist, so no imputation is performed.**

Justification:

* The authors documented the dataset as complete, and the measurement above
  confirms it: **0 missing cells out of 342,705**.
* Applying an imputation method (mean/median/mode/fill) to data that has no gaps
  would fabricate observations and distort the statistics that follow.
* Removing rows or columns would discard real information for no benefit.

This is a deliberate, evidence-based decision — not an oversight. The check is
kept in the notebook so that the claim remains verifiable if the dataset is ever
replaced.

## 10. Duplicate-Record Analysis

5,206 of the 11,055 records repeat a feature vector that already appears in
the file. The dataset gives no website identifier, so we cannot tell whether
such a row is the same site recorded twice or a different site with identical
measurements. We therefore keep all 11,055 records for the primary analysis
and only quantify the repeats here.

In [ ]:
df = df_raw.copy().reset_index(drop=True)
class_counts = df["Result"].value_counts().sort_index()
print("Class distribution (Full dataset, n=11,055):")
print(class_counts)

# Sensitivity analysis: Duplicate records
# (Note: These are repeated feature vectors, not confirmed identical websites)
df_dedup = df.drop_duplicates().reset_index(drop=True)
dup_count = int(df.duplicated().sum())
print(f"\nSensitivity analysis: {dup_count:,} duplicate rows detected.")
print(f"Deduplicated dataset size: {len(df_dedup):,}")

comparison = pd.DataFrame({
    "Full": class_counts,
    "Deduplicated": df_dedup["Result"].value_counts().sort_index(),
})
comparison.index = ["Phishing (-1)", "Legitimate (1)"]
display(comparison)

print("\nClass balance (Full) :",
      f"{class_counts[-1] / len(df) * 100:.2f}% phishing / {class_counts[1] / len(df) * 100:.2f}% legitimate")

**Interpretation.** 5,206 of 11,055 records (47.1%) repeat a feature vector
already present, spread across 5,849 distinct vectors. The repeats fall
mostly on the legitimate class (3,327 rows) rather than phishing (1,879), so
dropping them would move the class split from 44.31% / 55.69% to
51.62% / 48.38%. That shift is a reason for caution, not a correction: with
no website IDs we cannot know which rows are redundant. All primary
statistics below use the full 11,055 records. A deduplicated copy (5,849
rows) is kept alongside as a sensitivity check and is labelled as such
wherever it appears.

## 11. Data Type Verification and Correction

In [ ]:
print("DATA TYPES BEFORE CORRECTION")
print(df.dtypes.value_counts().to_string())
print()

# The dataset stores every column as int64, which is the correct type for
# integer-coded categorical data (compact, exact, no float rounding issues).
non_integer = [c for c in df.columns if not pd.api.types.is_integer_dtype(df[c])]
print("Columns that are not integer-typed:", non_integer if non_integer else "none")

# Explicitly enforce int8 (the values only need the range -1..1)
df = df.astype("int8")

print("\nDATA TYPES AFTER CORRECTION (int8 - exact, memory efficient)")
print(df.dtypes.value_counts().to_string())
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

**Interpretation.** All 31 columns are already integers, so no value-level
conversion was needed — there are no numbers hidden inside text and no dates
stored as strings (this dataset contains no date column at all). The only
worthwhile change is to narrow the dtype from `int64` to `int8`: the values are
restricted to `-1, 0, 1`, so `int8` stores them exactly while using one eighth of
the memory. This is a safe, reversible optimisation.

## 12. Column Renaming

Several original column names are misspelled, inconsistently capitalised or
ambiguous. Renaming them makes the analysis readable and prevents
misinterpretation.

In [ ]:
RENAME_MAP = {
    "having_IP_Address":          "Has_IP_Address",
    "URL_Length":                 "URL_Length",
    "Shortining_Service":         "Uses_Shortening_Service",   # original typo: "Shortining"
    "having_At_Symbol":           "Has_At_Symbol",
    "double_slash_redirecting":   "Double_Slash_Redirect",
    "Prefix_Suffix":              "Has_Prefix_Suffix",
    "having_Sub_Domain":          "Has_Sub_Domain",
    "SSLfinal_State":             "SSL_State",
    "Domain_registeration_length":"Domain_Registration_Length", # original typo: "registeration"
    "Favicon":                    "Favicon_From_Domain",
    "port":                       "Nonstandard_Port",
    "HTTPS_token":                "HTTPS_Token_In_Domain",
    "Request_URL":                "External_Request_URL",
    "URL_of_Anchor":              "External_Anchor_URL",
    "Links_in_tags":              "External_Links_In_Tags",
    "SFH":                        "Server_Form_Handler",
    "Submitting_to_email":        "Submits_To_Email",
    "Abnormal_URL":               "Abnormal_URL",
    "Redirect":                   "Uses_Redirect",
    "on_mouseover":               "Mouseover_Changes_Status",
    "RightClick":                 "RightClick_Disabled",
    "popUpWidnow":                "PopUp_Window",              # original typo: "Widnow"
    "Iframe":                     "Uses_Iframe",
    "age_of_domain":              "Domain_Age",
    "DNSRecord":                  "Has_DNS_Record",
    "web_traffic":                "Web_Traffic",
    "Page_Rank":                  "Page_Rank",
    "Google_Index":               "Google_Indexed",
    "Links_pointing_to_page":     "Links_Pointing_To_Page",
    "Statistical_report":         "In_Statistical_Report",
    "Result":                     "Class",
}

df = df.rename(columns=RENAME_MAP)

print(f"Columns renamed: {len(RENAME_MAP)}")
print("New column names:")
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c}")

**Interpretation.** Three names contained outright spelling errors
(`Shortining_Service`, `Domain_registeration_length`, `popUpWidnow`), several used
inconsistent capitalisation, and abbreviations such as `SFH` were expanded to
`Server_Form_Handler`. The target `Result` is renamed `Class` so its role is
unambiguous. All names are now clear, consistent and self-describing.

## 13. Invalid Value Detection

In [ ]:
FEATURES = [c for c in df.columns if c != "Class"]
ALLOWED = {-1, 0, 1}

print("VALUE-DOMAIN CHECK (every feature must stay within {-1, 0, 1})")
print("=" * 62)

violations = {}
for c in FEATURES:
    found = set(df[c].unique())
    if not found.issubset(ALLOWED):
        violations[c] = found

print("Features with out-of-range values:", violations if violations else "none")

print("\nTARGET CHECK")
print("Class values present:", sorted(df['Class'].unique()),
      "-> only -1 (phishing) and 1 (legitimate); no '0 / suspicious' class exists in this dataset")

print("\nLOGICAL CONSISTENCY CHECK")
print("Rows where Class is outside {-1, 1}:", int((~df['Class'].isin([-1, 1])).sum()))
print("Rows with all-zero features (no signal at all):",
      int((df[FEATURES] == 0).all(axis=1).sum()))

**Interpretation.** No invalid values were found: all 30 features are strictly
within `{-1, 0, 1}` and the target contains only `-1` and `1`. Nothing had to be
removed or corrected. The one structural observation worth recording is that the
dataset is **binary** — the "suspicious" (`0`) class that appears in some
descriptions of this dataset is not present in the target column.

## 14. Verification of the Analysis Dataset

In [ ]:
summary = pd.DataFrame({
    "Check": [
        "Rows", "Columns", "Missing values", "Duplicate rows",
        "Non-integer columns", "Out-of-range feature values",
        "Target classes", "Memory usage (KB)",
    ],
    "Result": [
        f"{len(df):,}", len(df.columns), int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
        len([c for c in df.columns if not pd.api.types.is_integer_dtype(df[c])]),
        len(violations),
        f"{sorted(df['Class'].unique())}",
        round(df.memory_usage(deep=True).sum() / 1024, 1),
    ],
})
display(summary)

display(df.describe().T.round(3))

## 15. Derived Variable — Risk Signal Count

Every feature is a three-level code, which is too coarse for distribution plots.
A single informative variable is therefore derived:

> **`Risk_Signals`** = number of the 30 features in a row whose value is `-1`
> (the phishing-leaning code).

This is a documented, transparent transformation: no information is invented, and
the variable is used only for distributional visualization (histogram, box plot,
violin plot, line chart). It is clearly labelled as derived wherever it appears.

In [ ]:
df["Risk_Signals"] = (df[FEATURES] == -1).sum(axis=1)
df["Class_Label"] = df["Class"].map({-1: "Phishing", 1: "Legitimate"})

print(df.groupby("Class_Label")["Risk_Signals"].describe().round(2))

print("\nOverall mean risk signals:",
      round(df['Risk_Signals'].mean(), 2),
      "| median:", int(df['Risk_Signals'].median()),
      "| min:", int(df['Risk_Signals'].min()),
      "| max:", int(df['Risk_Signals'].max()))

## 16. Descriptive Statistics

Required measures: **mean, median, mode, minimum, maximum, range, standard
deviation and variance** — computed for every feature.

In [ ]:
stats_rows = []
for c in FEATURES + ["Risk_Signals"]:
    s = df[c]
    stats_rows.append({
        "Feature": c,
        "Mean": round(s.mean(), 3),
        "Median": round(s.median(), 3),
        "Mode": s.mode().iloc[0],
        "Min": s.min(),
        "Max": s.max(),
        "Range": s.max() - s.min(),
        "Std_Dev": round(s.std(), 3),
        "Variance": round(s.var(), 3),
    })

stats_table = pd.DataFrame(stats_rows).set_index("Feature")
display(stats_table)

### 16.1 Mean

$$\bar{x} = \frac{\sum x}{n}$$

where $\sum x$ is the sum of all values and $n$ the number of observations.

In [ ]:
ssl = df["SSL_State"]
manual_mean = ssl.sum() / len(ssl)

print(f"SSL_State  ->  sum = {ssl.sum()}, n = {len(ssl)}")
print(f"Manual mean  = {ssl.sum()} / {len(ssl)} = {manual_mean:.4f}")
print(f"Pandas mean  = {ssl.mean():.4f}")
print()
print("Mean SSL_State by class:")
print(df.groupby("Class")["SSL_State"].mean().round(4).to_string())

**Interpretation.** The overall mean `SSL_State` is about `0.25`, i.e. slightly on
the legitimate side. Split by class, phishing sites average `-0.48` and
legitimate sites `0.83`. The sign difference is the signal: a negative mean means
the typical phishing site fails the SSL check, a positive mean means the typical
legitimate site passes it. Because the codes are ordinal, the mean is best read as
*net direction of evidence*, not as a physical quantity.

### 16.2 Median — and why it matters alongside the mean

In [ ]:
for c in ["SSL_State", "External_Anchor_URL", "Web_Traffic", "Risk_Signals"]:
    m, med = df[c].mean(), df[c].median()
    skew = "right-skewed" if m > med else ("left-skewed" if m < med else "symmetric")
    print(f"{c:22s} mean={m:7.3f}  median={med:7.3f}  -> {skew}")

**Interpretation.** For `Risk_Signals` the mean (8.90) sits slightly above the
median (8), which indicates a mild right skew: most sites carry a moderate number
of phishing-leaning signals, while a tail of sites carries many. In such a
distribution the **median is the more honest "typical" value** — the mean is pulled
upward by the tail. For the three-level codes the median usually equals the
middle code, which is why the mean is the more informative summary there.

### 16.3 Mode — most frequent value

In [ ]:
mode_table = pd.DataFrame({
    "Mode": [df[c].mode().iloc[0] for c in FEATURES],
    "Mode_Count": [int((df[c] == df[c].mode().iloc[0]).sum()) for c in FEATURES],
}, index=FEATURES)
mode_table["Mode_%"] = (mode_table["Mode_Count"] / len(df) * 100).round(1)
mode_table = mode_table.sort_values("Mode_%", ascending=False)

display(mode_table.head(12))

**Interpretation.** Mode is the natural summary for categorical codes. Several
features are dominated by a single value: `RightClick_Disabled` is `1` for 95.7%
of sites, `Uses_Iframe` is `1` for 90.8%, `Has_Prefix_Suffix` is `-1` for 86.7%,
and `Uses_Redirect` is `0` for 88.4%. A feature where one value covers most of the
data has little room left to discriminate — this is confirmed later by the
correlation analysis.

### 16.4 Standard Deviation and Variance — amount of variation

In [ ]:
variation = stats_table[["Std_Dev", "Variance"]].sort_values("Std_Dev", ascending=False)
display(variation.head(8))
print("\nLeast variable features:")
display(variation.tail(5))

**Interpretation.** `Domain_Age` has the largest standard deviation in the file
(1.00) because it is split fairly evenly between its two observed codes (`-1`:
5,189 sites, `1`: 5,866 sites — the middle level never occurs).
`Uses_Redirect` has the smallest (≈0.32) because 88.4% of sites share the same
value. High variation is a *precondition* for a feature to be useful, but it is
not sufficient: `Domain_Age` varies maximally yet correlates with the class at
only r ≈ 0.12, whereas `SSL_State` varies less (std ≈ 0.91) but correlates
strongly (r ≈ 0.71).

## 17. Correlation Analysis

Pearson correlation between each feature and the class label, plus the full
feature-to-feature matrix. Coefficients range from `-1` (perfect negative) through
`0` (no linear relationship) to `+1` (perfect positive).

In [ ]:
corr_with_class = (df[FEATURES + ["Class"]]
                   .corr()["Class"]
                   .drop("Class")
                   .sort_values(key=np.abs, ascending=False))

corr_df = pd.DataFrame({
    "Correlation_with_Class": corr_with_class.round(3),
    "Direction": np.where(corr_with_class > 0, "higher -> legitimate", "higher -> phishing"),
}).head(15)

display(corr_df)

In [ ]:
corr_matrix = df[FEATURES + ["Class"]].corr()

# strongest feature-to-feature relationships (excluding the diagonal)
pairs = (corr_matrix.where(~np.eye(len(corr_matrix), dtype=bool))
         .stack()
         .reset_index())
pairs.columns = ["Feature_A", "Feature_B", "Correlation"]
pairs["Abs"] = pairs["Correlation"].abs()
pairs = pairs[pairs["Feature_A"] < pairs["Feature_B"]].sort_values("Abs", ascending=False)

print("Strongest feature-to-feature correlations:")
display(pairs.head(10)[["Feature_A", "Feature_B", "Correlation"]].round(3))

**Interpretation of the key correlations**

| Feature | r with class | Meaning |
|---|---|---|
| `SSL_State` | **+0.71** | Strongest single signal. Valid SSL strongly indicates legitimacy. |
| `External_Anchor_URL` | **+0.69** | Anchor links pointing off-site are the second strongest signal. |
| `Has_Prefix_Suffix` | +0.35 | A `-` in the domain is common in phishing, but also in many legitimate sites. |
| `Web_Traffic` | +0.35 | Popular sites are usually legitimate. |
| `Has_Sub_Domain` | +0.30 | Deep sub-domain nesting leans phishing. |
| `External_Request_URL`, `External_Links_In_Tags` | +0.25 | Resources and links loaded from off-site. |
| `Domain_Registration_Length` | −0.23 | Short registration periods lean phishing. |
| `Server_Form_Handler` | +0.22 | Forms posting to blank/foreign handlers lean phishing. |
| `Google_Indexed`, `In_Statistical_Report`, `Has_IP_Address` | +0.08 … +0.13 | Weak. |
| `RightClick_Disabled`, `PopUp_Window`, `Uses_Iframe`, `Favicon_From_Domain` | ≈ 0.00 | No usable linear signal. |

The correlation matrix also shows that the strongest predictors are only
moderately correlated with each other, so they contribute **different**
information rather than duplicating one another.

## 18. Data Visualizations

The ten required chart types follow. Each chart has a title, labelled axes, an
appropriate figure size, and a short interpretation of what it shows and why it
matters.

### 18.1 Histogram — distribution of risk signals by class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

bins = np.arange(df["Risk_Signals"].min() - 0.5, df["Risk_Signals"].max() + 1.5, 1)
ax.hist(df.loc[df.Class == -1, "Risk_Signals"], bins=bins, alpha=0.65,
        label="Phishing", color="#c0392b", edgecolor="white")
ax.hist(df.loc[df.Class == 1, "Risk_Signals"], bins=bins, alpha=0.65,
        label="Legitimate", color="#2471a3", edgecolor="white")

ax.set_title("Histogram: distribution of phishing-leaning signals per website")
ax.set_xlabel("Number of features coded -1 (phishing-leaning)")
ax.set_ylabel("Number of websites")
ax.legend(title="Class")
save_figure(fig, "18_01_histogram_risk_signals")
plt.show()

print("Phishing sites   : mean {:.2f}, median {:.0f}".format(
    df.loc[df.Class == -1, "Risk_Signals"].mean(), df.loc[df.Class == -1, "Risk_Signals"].median()))
print("Legitimate sites : mean {:.2f}, median {:.0f}".format(
    df.loc[df.Class == 1, "Risk_Signals"].mean(), df.loc[df.Class == 1, "Risk_Signals"].median()))

**What it shows.** Two distributions with the same shape but a clear rightward
shift for phishing: the phishing median is 10 failed checks (IQR 8–12) against 7
for legitimate sites (IQR 5–10), and the phishing tail extends to 25 while the
legitimate tail stops at 19. The distributions **overlap substantially** — 66.1%
of phishing sites and 38.8% of legitimate sites fall in the same 7–11 band.

**Why it matters.** A naive "count how many checks failed" score is a weak-to-moderate
separator on its own. The strong separation observed in this dataset comes from
*specific* features (SSL state, external anchors), not from simply adding up all
30 checks — an important argument for feature selection rather than aggregation.

### 18.2 Bar Chart — ten strongest features by correlation strength

In [ ]:
top10 = corr_with_class.abs().head(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10.index, top10.values,
               color=["#c0392b" if corr_with_class[c] < 0 else "#2471a3" for c in top10.index])

for bar, val in zip(bars, top10.values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)

ax.set_title("Bar chart: absolute correlation of each feature with the class label")
ax.set_xlabel("|Pearson r| with class (-1 = phishing, 1 = legitimate)")
ax.set_ylabel("Feature")
ax.set_xlim(0, 0.8)
save_figure(fig, "18_02_bar_top_correlations")
plt.show()

**What it shows.** `SSL_State` (0.71) and `External_Anchor_URL` (0.69) are far ahead
of everything else; the curve drops sharply to 0.35 and below.

**Why it matters.** It identifies the small set of checks a detection system
should prioritise — and warns that roughly half the features carry almost no
signal.

### 18.3 Pie Chart — class balance of the full dataset

In [ ]:
counts = df["Class"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=["Phishing (-1)", "Legitimate (1)"],
    autopct=lambda p: f"{p:.2f}%\n({int(round(p * len(df) / 100)):,})",
    startangle=90,
    colors=["#c0392b", "#2471a3"],
    explode=(0.03, 0.03),
    textprops={"fontsize": 11},
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
ax.set_title(f"Pie chart: class distribution of the full dataset (n = {len(df):,})")
save_figure(fig, "18_03_pie_class_balance")
plt.show()

**What it shows.** In the full dataset of 11,055 records the classes split
44.31% phishing (4,898) and 55.69% legitimate (6,157).

**Why it matters.** Legitimate sites outnumber phishing ones by about 1.26 : 1.
That is a mild imbalance — enough to remember when reading accuracy later, not
enough to require resampling. For reference, the deduplicated sensitivity set
flips the split to 51.62% / 48.38%, which is one reason we do not treat
deduplication as the corrected view of the data.

### 18.4 Scatter Plot — SSL state vs external anchor URLs

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
jitter = 0.12
sample = df.sample(2000, random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(9, 6.5))
for cls, colour, label in [(-1, "#c0392b", "Phishing"), (1, "#2471a3", "Legitimate")]:
    sub = sample[sample.Class == cls]
    ax.scatter(sub["SSL_State"] + rng.normal(0, jitter, len(sub)),
               sub["External_Anchor_URL"] + rng.normal(0, jitter, len(sub)),
               s=14, alpha=0.45, c=colour, label=label, edgecolors="none")

ax.set_title("Scatter plot: SSL state vs external anchor URL (2,000-site sample)")
ax.set_xlabel("SSL_State   (-1 invalid / 0 suspicious / 1 valid certificate)")
ax.set_ylabel("External_Anchor_URL   (-1 off-site / 0 mixed / 1 on-site)")
ax.set_xticks([-1, 0, 1])
ax.set_yticks([-1, 0, 1])
ax.legend(title="Class", markerscale=2)
save_figure(fig, "18_04_scatter_ssl_vs_anchor")
plt.show()

**What it shows.** Points are jittered slightly because both axes are three-level
codes. The bottom-left corner (invalid SSL, off-site anchors) is dominated by
phishing; the top-right corner (valid SSL, on-site anchors) is dominated by
legitimate sites.

**Why it matters.** Two features together separate the classes better than either
alone — a visual argument for combining checks in a detection rule.

### 18.5 Line Chart — detection rate as the risk threshold changes

In [ ]:
thresholds = np.arange(0, df["Risk_Signals"].max() + 1)
phish_rate = [(df.loc[df.Class == -1, "Risk_Signals"] >= t).mean() * 100 for t in thresholds]
legit_rate = [(df.loc[df.Class == 1, "Risk_Signals"] >= t).mean() * 100 for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(thresholds, phish_rate, marker="o", linewidth=2.2, color="#c0392b", label="Phishing caught")
ax.plot(thresholds, legit_rate, marker="s", linewidth=2.2, color="#2471a3", label="Legitimate flagged")

ax.set_title("Line chart: share of each class at or above a risk-signal threshold")
ax.set_xlabel("Risk-signal threshold (number of -1 codes)")
ax.set_ylabel("Share of class (%)")
ax.grid(True, alpha=0.4)
ax.legend()
save_figure(fig, "18_05_line_threshold_rates")
plt.show()

for t in (8, 10, 12):
    p = (df.loc[df.Class == -1, "Risk_Signals"] >= t).mean() * 100
    l = (df.loc[df.Class == 1, "Risk_Signals"] >= t).mean() * 100
    print(f"threshold {t:2d}: {p:5.1f}% of phishing flagged, {l:5.1f}% of legitimate flagged")

**What it shows.** Both curves decline as the threshold rises, but at different
rates. The gap is widest at a threshold of 8 (82.7% of phishing sites flagged
versus 42.0% of legitimate sites) and narrows sharply after 12 (26.6% versus
16.2%). At a strict threshold of 15 only 13.5% of phishing sites remain, but just
2.3% of legitimate sites are wrongly flagged.

**Why it matters.** It converts an abstract score into an operational trade-off:
loosening the threshold catches more phishing at the cost of more false alarms,
and tightening it does the reverse. No threshold on this score alone is
simultaneously high-recall and low-false-positive — which is why the specific
high-correlation features matter more than the aggregate count.

### 18.6 Box Plot — risk signals by class

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.boxplot(data=df, x="Class_Label", y="Risk_Signals",
            order=["Phishing", "Legitimate"],
            hue="Class_Label", legend=False,
            palette={"Phishing": "#c0392b", "Legitimate": "#2471a3"},
            width=0.5, ax=ax, showmeans=True,
            meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black"})
ax.set_title("Box plot: distribution of phishing-leaning signals by class")
ax.set_xlabel("Class")
ax.set_ylabel("Number of -1 coded features")
save_figure(fig, "18_06_box_risk_signals")
plt.show()

print(df.groupby("Class_Label")["Risk_Signals"].agg(["min", "median", "max", "std"]).round(2).to_string())

**What it shows.** The two boxes **overlap heavily** — phishing spans an
interquartile range of 8–12 and legitimate sites 5–10 — but their centres are
distinct: median 10 versus 7. The phishing class also has the longer upper tail
(maximum 25 failed checks, versus 19 for legitimate sites).

**Why it matters.** Box plots expose spread and outliers, not just averages. The
overlap is the honest result: many legitimate sites fail a similar number of checks
as phishing sites, so a count-based rule alone cannot be trusted. The long phishing
tail shows that some phishing sites are engineered to pass almost every check.

### 18.7 Correlation Heatmap

In [ ]:
top_features = corr_with_class.abs().head(12).index.tolist()
heat_data = df[top_features + ["Class"]].corr()

fig, ax = plt.subplots(figsize=(11, 8.5))
sns.heatmap(heat_data, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Correlation heatmap: 12 strongest features and the class label")
ax.set_xlabel("Feature")
ax.set_ylabel("Feature")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
save_figure(fig, "18_07_heatmap_correlations")
plt.show()

**What it shows.** Strong red/blue cells along the `Class` row identify the
predictive features. Between features themselves most cells are pale, meaning the
top predictors are largely independent.

**Why it matters.** Low inter-feature correlation is good news for modelling:
the strong signals add up instead of duplicating each other.

### 18.8 Count Plot — sites per SSL state, split by class

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.countplot(data=df, x="SSL_State", hue="Class_Label",
              hue_order=["Phishing", "Legitimate"],
              palette={"Phishing": "#c0392b", "Legitimate": "#2471a3"}, ax=ax)

ax.set_title("Count plot: number of websites per SSL state, by class")
ax.set_xlabel("SSL_State   (-1 invalid / 0 suspicious / 1 valid)")
ax.set_ylabel("Number of websites")
ax.legend(title="Class")
save_figure(fig, "18_08_countplot_ssl_state")
plt.show()

print(pd.crosstab(df["SSL_State"], df["Class"], margins=True).to_string())

**What it shows.** Of 4,898 phishing sites, 3,051 (62.3%) have an invalid SSL
state, compared with 506 of 6,157 legitimate sites (8.2%). Conversely 5,630
legitimate sites (91.4%) have a valid certificate.

**Why it matters.** SSL state is the single most actionable check: it is cheap to
compute for any URL and it separates the classes strongly.

### 18.9 Pair Plot — four strongest features plus the derived score

In [ ]:
pair_cols = ["SSL_State", "External_Anchor_URL", "Web_Traffic", "Risk_Signals"]
pair_sample = df.sample(1500, random_state=RANDOM_STATE).copy()
pair_sample["Class_Label"] = pair_sample["Class"].map({-1: "Phishing", 1: "Legitimate"})

g = sns.pairplot(pair_sample[pair_cols + ["Class_Label"]],
                 hue="Class_Label",
                 palette={"Phishing": "#c0392b", "Legitimate": "#2471a3"},
                 diag_kind="kde", plot_kws={"s": 12, "alpha": 0.45})
g.fig.suptitle("Pair plot: strongest features and the derived risk score", y=1.02, fontsize=13, fontweight="bold")
g.fig.set_size_inches(10, 9)
save_figure(g.fig, "18_09_pairplot_features")
plt.show()

**What it shows.** On the diagonal, the class-specific densities overlap only
partially. In the off-diagonal panels, `SSL_State` and `External_Anchor_URL` show
the clearest separation between the two colours, while `Risk_Signals` — being an
aggregate of all 30 checks — mixes the classes more than either of them.

**Why it matters.** Pair plots reveal whether two features interact — for example
whether phishing sites with a valid certificate compensate elsewhere. They also
demonstrate that a few well-chosen features separate the classes better than a
summary of all of them.

### 18.10 Violin Plot — risk-signal density by class

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.violinplot(data=df, x="Class_Label", y="Risk_Signals",
               order=["Phishing", "Legitimate"],
               hue="Class_Label", legend=False,
               palette={"Phishing": "#c0392b", "Legitimate": "#2471a3"},
               inner="quartile", cut=0, ax=ax)
ax.set_title("Violin plot: density of phishing-leaning signals by class")
ax.set_xlabel("Class")
ax.set_ylabel("Number of -1 coded features")
save_figure(fig, "18_10_violin_risk_signals")
plt.show()

**What it shows.** A violin plot is a mirrored, smoothed histogram. The phishing
violin bulges around 9 failed checks, the legitimate violin around 6, and both
taper into the same middle band.

**Why it matters.** It shows the *shape* of the distribution rather than a single
summary number. Phishing sites are not uniformly "bad": they cluster at a
particular level of failed checks, and the visible overlap with the legitimate
distribution sets a realistic expectation for any threshold-based rule.

## 19. Insights and Findings

Each insight below is computed from the dataset in the preceding cells — no
insight is asserted without a number behind it.

In [ ]:
phish = df[df.Class == -1]
legit = df[df.Class == 1]

print(f"Full dataset         : {len(df):,} websites")
print(f"Phishing             : {len(phish):,} ({len(phish)/len(df)*100:.2f}%)")
print(f"Legitimate           : {len(legit):,} ({len(legit)/len(df)*100:.2f}%)")
print()
print("SSL_State = -1  ->  phishing: {:.1f}%  |  legitimate: {:.1f}%".format(
    (phish.SSL_State == -1).mean() * 100, (legit.SSL_State == -1).mean() * 100))
print("External_Anchor_URL = -1  ->  phishing: {:.1f}%  |  legitimate: {:.1f}%".format(
    (phish.External_Anchor_URL == -1).mean() * 100, (legit.External_Anchor_URL == -1).mean() * 100))
print("Has_Prefix_Suffix = -1  ->  phishing: {:.1f}%  |  legitimate: {:.1f}%".format(
    (phish.Has_Prefix_Suffix == -1).mean() * 100, (legit.Has_Prefix_Suffix == -1).mean() * 100))
print("Web_Traffic = 1  ->  phishing: {:.1f}%  |  legitimate: {:.1f}%".format(
    (phish.Web_Traffic == 1).mean() * 100, (legit.Web_Traffic == 1).mean() * 100))
print("RightClick_Disabled = 1  ->  phishing: {:.1f}%  |  legitimate: {:.1f}%".format(
    (phish.RightClick_Disabled == 1).mean() * 100, (legit.RightClick_Disabled == 1).mean() * 100))
print()
print("Mean risk signals  ->  phishing: {:.2f}  |  legitimate: {:.2f}".format(
    phish.Risk_Signals.mean(), legit.Risk_Signals.mean()))

### Twelve evidence-based insights

**1. Nearly half the records repeat a feature vector, and that is a caution, not a fix.**
5,206 of 11,055 rows (47.1%) repeat a row already present. With no website
IDs we cannot call them duplicate sites, so the primary analysis keeps all
11,055 records. A deduplicated copy (5,849 rows) is used only as a
sensitivity check; it moves the class split from 44.31% / 55.69% to
51.62% / 48.38%, because the repeats fall mostly on the legitimate class
(3,327 rows against 1,879).

**2. The full dataset leans legitimate, but not severely.**
4,898 phishing (44.31%) versus 6,157 legitimate (55.69%) — a ratio of about
1 : 1.26. Worth remembering when reading accuracy, not worth resampling.

**3. There are no missing values at all.**
0 missing cells out of 342,705. Imputation would therefore have fabricated
data; the correct decision was to perform none.

**4. SSL state is the strongest single indicator of legitimacy (r = 0.71).**
62.3% of phishing sites have an invalid or absent certificate, against only
8.2% of legitimate sites. 91.4% of legitimate sites present a valid one.

**5. External anchor URLs are the strongest indicator of phishing (r = 0.69).**
66.3% of phishing sites have anchor links pointing off-site, versus 0.6% of
legitimate sites — effectively a near-perfect separator on its own.

**6. A hyphen in the domain is a necessary but not sufficient phishing signal.**
100% of phishing sites are coded `-1` for `Has_Prefix_Suffix`, but so are 76.2%
of legitimate sites. It cannot be used alone; its correlation is only 0.35.

**7. Popularity is a real, measurable protective factor.**
70.2% of legitimate sites show positive web traffic against 30.8% of phishing
sites (r = 0.35). Newly created phishing domains have not yet accumulated
visitors.

**8. Several features carry no usable signal and should be dropped in modelling.**
`RightClick_Disabled` (95.4% vs 95.9% coded `1`), `PopUp_Window` (r = +0.00),
`Uses_Iframe` (r = −0.00) and `Favicon_From_Domain` (r = −0.00) behave almost
identically in both classes. In total **18 of the 30 features correlate below
|r| = 0.10**, and 11 of those below |r| = 0.05.

**9. High variance does not imply predictive power.**
`Domain_Age` has the largest standard deviation in the dataset (≈1.00, split
5,189 / 5,866 between its two observed codes) yet correlates with the class at
only r ≈ 0.12. Variation is a necessary condition for a useful feature, not a
sufficient one.

**10. A simple count of failed checks is only a weak-to-moderate separator.**
Mean risk signals are 10.41 for phishing versus 7.71 for legitimate sites —
a gap of 2.7 checks — and 66.1% of phishing sites fall in the same 7–11 band
as 38.8% of legitimate sites. Aggregating all 30 checks is therefore much
less informative than using the handful of features with strong individual
correlation.

**11. Short domain registration periods lean phishing.**
`Domain_Registration_Length` correlates negatively (r = −0.23): phishing
domains average −0.10 versus −0.53 for legitimate domains, i.e. shorter
registration commitments.

**12. The target is binary — the "suspicious" class is absent.**
The target contains only `-1` and `1`; no `0` class exists, even though some
descriptions of this dataset mention one. Any later classifier should be
planned as a two-class problem.

## 20. Conclusion

This analysis examined the UCI Phishing Websites dataset — 11,055 websites
described by 30 integer-coded security features. The first substantive finding
was about the data itself: **47.1% of the records repeat a feature vector**
already present. Because the file carries no website identifiers, those rows
cannot be proven to be duplicate sites, so the primary analysis keeps all
11,055 records and reports 44.31% phishing / 55.69% legitimate. A
deduplicated copy (5,849 rows) is reported separately as a sensitivity check;
it shifts the split to 51.62% / 48.38%, which is exactly why it is not treated
as the corrected dataset. There were no missing values and no out-of-range
codes, so the honest cleaning decisions were to narrow the integer types,
correct three misspelled column names, and document that no imputation was
required rather than applying one for appearance.

Statistically, phishing detection from URL and page structure looks highly
feasible: `SSL_State` (r = 0.71) and `External_Anchor_URL` (r = 0.69) are
strong, largely independent predictors, followed by domain-structure features
at r ≈ 0.22–0.35. At the same time, **18 of the 30 features correlate below
|r| = 0.10** (11 of them below 0.05), and high variance alone — as in
`Domain_Age`, with the maximum possible standard deviation but r ≈ 0.12 — does
not make a feature predictive. A naive count of failed checks separated the
classes only moderately (means 10.41 versus 7.71, with heavy overlap), which
underlines that *selecting the right features matters far more than
aggregating many of them*.

The practical value is direct: these are cheap, automatable checks that a
browser, mail gateway or SOC triage pipeline can compute for any URL. The
analysis also defines the groundwork for the next stage — the seven features
with |r| ≥ 0.24 (`SSL_State`, `External_Anchor_URL`, `Has_Prefix_Suffix`,
`Web_Traffic`, `Has_Sub_Domain`, `External_Request_URL`,
`External_Links_In_Tags`) form a natural candidate set for a baseline
classifier, while the near-zero-correlation features can be dropped to reduce
noise.

**What was learned:** that most of the effort in a data project goes into
verifying and cleaning the data before analysis, that statistical summaries
must be interpreted in the context of the encoding, and that correlation alone
is not evidence of predictive value.

## 21. References

1. Mohammad, R. M., & McCluskey, L. (2012). *Phishing Websites* [Dataset]. UCI
   Machine Learning Repository. https://doi.org/10.24432/C51W2X (CC BY 4.0)
2. Mohammad, R. M., Thabtah, F., & McCluskey, L. (2012). *An assessment of features
   related to phishing websites using an automated technique.* International
   Conference for Internet Technology and Secured Transactions (ICITST).
3. UCI Machine Learning Repository — dataset page:
   https://archive.ics.uci.edu/dataset/327/phishing+websites
4. Kaggle mirror used for the Kaggle-sourced requirement:
   https://www.kaggle.com/datasets/akashkr/phishing-website-dataset
5. Pandas documentation — https://pandas.pydata.org/docs/
6. NumPy documentation — https://numpy.org/doc/
7. Matplotlib documentation — https://matplotlib.org/stable/
8. Seaborn documentation — https://seaborn.pydata.org/
9. OWASP Top 10 (2021) — A07: Identification and Authentication Failures
   (phishing context) — https://owasp.org/Top10/

---

### Reproducibility

* Random seed `42` is used for every sampling operation.
* The dataset is committed at `data/phishing_dataset.csv` (CC BY 4.0, attribution
  retained); the loader falls back to the official UCI archive if it is missing.
* Environment: Jupyter Notebook (Python 3.x), pandas, numpy, matplotlib, seaborn.
* Every number quoted in the insights and conclusion is printed by the cells above.

In [ ]:
import shutil

print(f"{len(_saved_figures)} figures saved to '{FIG_DIR}/'")
for f in _saved_figures:
    print("  ", f)

archive = shutil.make_archive("task01_visualizations", "zip", FIG_DIR)
print("\nZIP created:", archive)

try:
    from google.colab import files
    files.download(archive)
    print("Download started. Also download the notebook itself for the report screenshots.")
except ImportError:
    print("Not running in Colab - the ZIP is available locally.")


---
### Next steps after running this notebook

1. **Run all cells top to bottom** (the whole notebook runs in under a minute).
2. The last cell saves all ten charts to `visualizations/` and creates
   `task01_visualizations.zip` — download it for the report.
3. Take the required screenshots: code cells (loading, cleaning, EDA, statistics,
   visualizations) and their outputs.
4. The numbers printed in the output are the evidence base for the report; quote
   them exactly as printed.
